In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
import joblib

# تحميل البيانات
df = pd.read_csv("Final_Updates.csv")

# تحويل عمود 'Datetime' إلى نوع datetime
df['Datetime'] = pd.to_datetime(df['Datetime'])

# استخراج الميزات الزمنية
df['Month'] = df['Datetime'].dt.month
df['Day'] = df['Datetime'].dt.day
df['Hour'] = df['Datetime'].dt.hour
df['Minute'] = df['Datetime'].dt.minute
df['Weekday'] = df['Datetime'].dt.day_name()

# ترميز الأعمدة الفئوية
label_encoder_city = LabelEncoder()
df['City'] = label_encoder_city.fit_transform(df['City'])

label_encoder_congestion = LabelEncoder()
df['CongestionLevel'] = label_encoder_congestion.fit_transform(df['CongestionLevel'])

label_encoder_weekday = LabelEncoder()
df['Weekday'] = label_encoder_weekday.fit_transform(df['Weekday'])


In [2]:
# الميزات
features = [
    'TrafficIndexLive_norm', 'JamsCount_norm', 'TrafficIndexWeekAgo_norm',
    'TravelTimeHistoric_norm', 'TravelTimeLive_norm',
    'Hour', 'Weekday', 'IsWeekend', 'Year', 'Month', 'Day', 'Minute'
]

X = df[features]
y = df['CongestionLevel']

# تقسيم البيانات
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [3]:
# تدريب النموذج
logreg_model = LogisticRegression(solver='lbfgs', max_iter=8000, random_state=42)

logreg_model.fit(X_train, y_train)

# حفظ النموذج
joblib.dump(logreg_model, 'logreg_congestion_model.pkl')

# التنبؤ
y_pred = logreg_model.predict(X_test)

# الدقة والتقرير
accuracy = accuracy_score(y_test, y_pred)
target_names = label_encoder_congestion.classes_
classification_report_result = classification_report(y_test, y_pred, target_names=target_names.astype(str))

print(f"Model Accuracy: {accuracy}")
print("Classification Report:")
print(classification_report_result)

Model Accuracy: 0.8648033126293996
Classification Report:
              precision    recall  f1-score   support

        High       0.92      0.97      0.94      4035
         Low       0.00      0.00      0.00       239
    Moderate       0.47      0.47      0.47       556

    accuracy                           0.86      4830
   macro avg       0.46      0.48      0.47      4830
weighted avg       0.82      0.86      0.84      4830



C:\Users\Khaled-PC\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\Khaled-PC\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Khaled-PC\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 

In [ ]:
# تحميل النموذج
logreg_model = joblib.load('logreg_congestion_model.pkl')

trained_features = features  # نفسها مثل أعلاه

# التطبيع
def normalize_input(user_input):
    user_input['TrafficIndexLive_norm'] = user_input['TrafficIndexLive'] / 99
    user_input['JamsCount_norm'] = user_input['JamsCount'] / 883
    user_input['TrafficIndexWeekAgo_norm'] = user_input['TrafficIndexWeekAgo'] / 99
    user_input['TravelTimeHistoric_norm'] = user_input['TravelTimeHistoric'] / 94.7783212767502
    user_input['TravelTimeLive_norm'] = user_input['TravelTimeLive'] / 130.973103722656
    return user_input

# إدخال المستخدم
def get_user_input():
    return {
        'TrafficIndexLive': float(input("TrafficIndexLive: ")),
        'JamsCount': int(input("JamsCount: ")),
        'TrafficIndexWeekAgo': float(input("TrafficIndexWeekAgo: ")),
        'TravelTimeHistoric': float(input("TravelTimeHistoric: ")),
        'TravelTimeLive': float(input("TravelTimeLive: ")),
        'Hour': int(input("Hour (0-23): ")),
        'Weekday': int(input("Weekday (0=Monday to 6=Sunday): ")),
        'IsWeekend': int(input("IsWeekend (0 or 1): ")),
        'Year': int(input("Year: ")),
        'Month': int(input("Month: ")),
        'Day': int(input("Day: ")),
        'Minute': int(input("Minute: "))
    }

# التنبؤ
def predict_congestion(user_input):
    user_input = normalize_input(user_input)
    input_filtered = {key: user_input[key] for key in trained_features}
    user_input_df = pd.DataFrame([input_filtered], columns=trained_features)

    prediction = logreg_model.predict(user_input_df)
    predicted_label = label_encoder_congestion.inverse_transform(prediction)
    return predicted_label[0]

# الاستخدام
user_input = get_user_input()
result = predict_congestion(user_input)
print(f"Predicted Congestion Level: {result}")